# 🤖 Alura Agente — Agente de IA con RAG sobre documentos (BimBam Buy)

Este notebook cubre las **tres etapas** del Challenge:

1. **Lectura y procesamiento** de documentos PDF (políticas de BimBam Buy).
2. **Agente de IA (RAG)** con LangChain + Google Gemini que responde preguntas en lenguaje natural.
3. **Preparación para el deploy** en OCI (se genera un `app.py` con FastAPI listo para subir a OCI Compute).

> Sugerencia del challenge: primero probamos todo **localmente en Colab**. El deploy en OCI se hace después, ya con el agente funcionando.


## 1️⃣ Instalación de dependencias

In [ ]:
!pip install -q \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    langchain-text-splitters==0.3.2 \
    langchain-google-genai==2.0.7 \
    pypdf==5.1.0 \
    faiss-cpu==1.9.0 \
    google-generativeai==0.8.3


⚠️ **Muy importante:** después de que esta celda termine de instalar, andá a **Entorno de ejecución → Reiniciar sesión** y volvé a ejecutar el notebook desde el principio (incluyendo esta celda de instalación). Esto es porque Python necesita "recargar" las librerías nuevas — si no reiniciás, vas a seguir viendo errores de `ModuleNotFoundError` aunque la instalación haya sido exitosa.


## 2️⃣ Configurar la API Key

Guardá tu clave en **Colab → 🔑 Secrets (barra lateral izquierda)** con el nombre **`APIKEY`** y activá el acceso al notebook.

Conseguí tu API Key gratis en: https://aistudio.google.com/app/apikey


In [ ]:
from google.colab import userdata
import os

# Nombre de la clave configurada en los Secrets de Colab: "APIKEY"
GOOGLE_API_KEY = userdata.get('APIKEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("✅ API Key cargada correctamente" if GOOGLE_API_KEY else "❌ No se encontró la API Key")


## 3️⃣ Subir los documentos (PDFs de BimBam Buy)

Subí acá los archivos PDF que va a usar el agente (por ejemplo, los de BimBam Buy: Política de Reembolsos, Programa de Afiliados, Guía de Envíos, Manual de Garantía, Preguntas Frecuentes de Métodos de Pago).

> 💡 También podés usar `from google.colab import drive; drive.mount('/content/drive')` si preferís leer los archivos desde tu Google Drive.


In [ ]:
from google.colab import files
import os

os.makedirs("documentos", exist_ok=True)
uploaded = files.upload()  # Seleccioná uno o varios PDFs

for filename in uploaded.keys():
    os.rename(filename, os.path.join("documentos", filename))

print("📂 Archivos cargados:")
for f in os.listdir("documentos"):
    print(" -", f)


## 4️⃣ Etapa 1: Leer y procesar los documentos

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Carga todos los PDFs de la carpeta "documentos"
loader = PyPDFDirectoryLoader("documentos")
documentos = loader.load()

print(f"📄 Se cargaron {len(documentos)} páginas de {len(uploaded)} documento(s).")

# Dividimos el contenido en fragmentos (chunks) para que el modelo
# pueda procesarlos y encontrar la información relevante.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = splitter.split_documents(documentos)

print(f"✂️ Se generaron {len(chunks)} fragmentos de texto.")


### (Opcional) Leer un CSV en lugar de / además de PDFs

Si tu documento es un CSV (por ejemplo, datos de ventas o el `inventario_de_supermercado_latam.xlsx`), usá esta celda en vez de la anterior.


In [ ]:
# from langchain_community.document_loaders.csv_loader import CSVLoader
#
# loader_csv = CSVLoader(file_path="documentos/tu_archivo.csv", encoding="utf-8")
# documentos_csv = loader_csv.load()
# chunks += splitter.split_documents(documentos_csv)


## 5️⃣ Etapa 2: Construir el agente de IA (RAG)

Creamos los *embeddings* (representación numérica del texto), los guardamos en una base vectorial (FAISS) y armamos la cadena de pregunta-respuesta con **Gemini**.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 1. Embeddings: convierten el texto en vectores numéricos
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. Base vectorial: la creamos en LOTES PEQUEÑOS con pausas, para no
#    superar el límite gratuito de la API (100 pedidos de embeddings por minuto)
import time

def crear_vectorstore_por_lotes(chunks, embeddings, tamano_lote=15, espera_segundos=20):
    vectorstore = None
    total_lotes = (len(chunks) // tamano_lote) + 1
    for i in range(0, len(chunks), tamano_lote):
        lote = chunks[i:i + tamano_lote]
        numero_lote = (i // tamano_lote) + 1
        intentos = 0
        while intentos < 5:
            try:
                if vectorstore is None:
                    vectorstore = FAISS.from_documents(lote, embeddings)
                else:
                    vectorstore.add_documents(lote)
                print(f"✅ Lote {numero_lote}/{total_lotes} procesado ({len(lote)} fragmentos)")
                break
            except Exception as e:
                intentos += 1
                print(f"⏳ Límite de la API alcanzado, esperando {espera_segundos}s antes de reintentar... (intento {intentos})")
                time.sleep(espera_segundos)
        time.sleep(3)  # pequeña pausa entre lotes, aunque hayan salido bien
    return vectorstore

vectorstore = crear_vectorstore_por_lotes(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 3. Modelo de lenguaje (LLM) que genera las respuestas
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
)

# 4. Prompt: le indicamos al modelo que responda SOLO en base a los documentos
prompt_template = PromptTemplate(
    template=(
        "Eres el asistente virtual de BimBam Buy. Respondé la pregunta del usuario "
        "utilizando ÚNICAMENTE la información del siguiente contexto extraído de los "
        "documentos oficiales de la empresa. Si la respuesta no está en el contexto, "
        "decí claramente que no tenés esa información.\n\n"
        "Contexto:\n{context}\n\n"
        "Pregunta: {question}\n\n"
        "Respuesta clara y concisa:"
    ),
    input_variables=["context", "question"],
)

# 5. Cadena RAG (Retrieval-Augmented Generation)
agente = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True,
)

print("✅ Agente listo para responder preguntas.")


## 6️⃣ Función para hacer preguntas al agente

In [ ]:
def preguntar(pregunta: str):
    resultado = agente.invoke({"query": pregunta})
    print("🙋 Pregunta:", pregunta)
    print("🤖 Respuesta:", resultado["result"])
    print("\n📚 Fuentes utilizadas:")
    for doc in resultado["source_documents"]:
        origen = doc.metadata.get("source", "desconocido")
        pagina = doc.metadata.get("page", "?")
        print(f"   - {origen} (página {pagina})")
    print("-" * 60)
    return resultado["result"]


## 7️⃣ Probar el agente con preguntas de ejemplo

Ajustá estas preguntas según los documentos que subiste (reembolsos, envíos, afiliados, garantía, métodos de pago, etc.).


In [ ]:
preguntar("¿Cuál es la política de reembolsos de BimBam Buy?")


In [ ]:
preguntar("¿Cómo funciona el programa de afiliados?")


In [ ]:
preguntar("¿Cuánto tarda el envío y cuánto cuesta?")


In [ ]:
preguntar("¿Qué cubre la garantía de los productos?")


## 8️⃣ Modo interactivo (opcional)

Ejecutá esta celda para hacerle preguntas al agente desde la consola de Colab.


In [ ]:
while True:
    pregunta = input("Escribí tu pregunta (o 'salir' para terminar): ")
    if pregunta.lower() in ("salir", "exit", "quit"):
        break
    preguntar(pregunta)


## 9️⃣ Etapa 3: Preparar el deploy en OCI

Una vez que el agente funciona bien localmente, generamos un pequeño servidor **FastAPI** que expone el agente como una API web. Este archivo (`app.py`) es el que después se sube y se ejecuta en una instancia de **OCI Compute**.

> Esta celda solo genera el archivo. El deploy real se hace conectándote por SSH a tu instancia OCI, subiendo estos archivos y ejecutando `python3 app.py` (o con `uvicorn`/`gunicorn` + un proceso en segundo plano).


In [ ]:
app_code = '''
import os
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# La API key se lee de una variable de entorno en el servidor OCI
os.environ["GOOGLE_API_KEY"] = os.environ.get("APIKEY", "")

app = FastAPI(title="Alura Agente - BimBam Buy")

loader = PyPDFDirectoryLoader("documentos")
documentos = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documentos)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

import time

def crear_vectorstore_por_lotes(chunks, embeddings, tamano_lote=15, espera_segundos=20):
    vectorstore = None
    for i in range(0, len(chunks), tamano_lote):
        lote = chunks[i:i + tamano_lote]
        intentos = 0
        while intentos < 5:
            try:
                if vectorstore is None:
                    vectorstore = FAISS.from_documents(lote, embeddings)
                else:
                    vectorstore.add_documents(lote)
                break
            except Exception:
                intentos += 1
                time.sleep(espera_segundos)
        time.sleep(3)
    return vectorstore

vectorstore = crear_vectorstore_por_lotes(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

prompt_template = PromptTemplate(
    template=(
        "Eres el asistente virtual de BimBam Buy. Respondé usando SOLO el contexto.\\n\\n"
        "Contexto:\\n{context}\\n\\nPregunta: {question}\\n\\nRespuesta:"
    ),
    input_variables=["context", "question"],
)

agente = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True,
)

class Pregunta(BaseModel):
    pregunta: str

@app.get("/")
def home():
    return {"status": "ok", "mensaje": "Alura Agente funcionando en OCI 🚀"}

@app.post("/preguntar")
def responder(payload: Pregunta):
    resultado = agente.invoke({"query": payload.pregunta})
    fuentes = [d.metadata.get("source") for d in resultado["source_documents"]]
    return {"respuesta": resultado["result"], "fuentes": fuentes}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

requirements = """fastapi
uvicorn
langchain==0.3.7
langchain-community==0.3.7
langchain-text-splitters==0.3.2
langchain-google-genai==2.0.7
pypdf==5.1.0
faiss-cpu==1.9.0
google-generativeai==0.8.3
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("✅ Se generaron app.py y requirements.txt, listos para subir a tu instancia OCI Compute.")


In [ ]:
from google.colab import files
files.download("app.py")
files.download("requirements.txt")


## 🔟 Checklist de entregables (README)

Recordá que tu repositorio en GitHub debe incluir:

- [ ] Código organizado + historial de commits.
- [ ] `README.md` con: descripción del proyecto, arquitectura (PDF → chunks → embeddings → FAISS → Gemini → respuesta), tecnologías usadas, instrucciones de ejecución, ejemplos de preguntas y respuestas.
- [ ] Este notebook (o el `.py` equivalente) que lee los documentos y responde preguntas.
- [ ] `app.py` + evidencia (enlace o captura) de que corre en OCI Compute.

¡Con esto ya tenés las tres etapas cubiertas: lectura del documento, agente funcional y base para el deploy! 🚀
